# Handling Imbalanced Data Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Generate an imbalanced dataset

In [ ]:
```python

import numpy as np

def make_imbalanced_data(n_majority=950, n_minority=50, seed=42):

    rng = np.random.RandomState(seed)

    X_maj = rng.randn(n_majority, 2) * 1.0 + np.array([0.0, 0.0])

    X_min = rng.randn(n_minority, 2) * 0.8 + np.array([2.5, 2.5])

    X = np.vstack([X_maj, X_min])

    y = np.concatenate([np.zeros(n_majority), np.ones(n_minority)])

    shuffle_idx = rng.permutation(len(y))

    return X[shuffle_idx], y[shuffle_idx]

In [ ]:
```

### Step 2: SMOTE from scratch

In [ ]:
```python

def euclidean_distance(a, b):

    return np.sqrt(np.sum((a - b) ** 2))

def find_k_neighbors(X, idx, k):

    distances = []

    for i in range(len(X)):

        if i == idx:

            continue

        d = euclidean_distance(X[idx], X[i])

        distances.append((i, d))

    distances.sort(key=lambda x: x[1])

    return [d[0] for d in distances[:k]]

def smote(X_minority, k=5, n_synthetic=100, seed=42):

    rng = np.random.RandomState(seed)

    n_samples = len(X_minority)

    k = min(k, n_samples - 1)

    synthetic = []

    for _ in range(n_synthetic):

        idx = rng.randint(0, n_samples)

        neighbors = find_k_neighbors(X_minority, idx, k)

        neighbor_idx = neighbors[rng.randint(0, len(neighbors))]

        t = rng.random()

        new_point = X_minority[idx] + t * (X_minority[neighbor_idx] - X_minority[idx])

        synthetic.append(new_point)

    return np.array(synthetic)

In [ ]:
```

### Step 3: Random oversampling and undersampling

In [ ]:
```python

def random_oversample(X, y, seed=42):

    rng = np.random.RandomState(seed)

    classes, counts = np.unique(y, return_counts=True)

    max_count = counts.max()

    X_resampled = list(X)

    y_resampled = list(y)

    for cls, count in zip(classes, counts):

        if count < max_count:

            cls_indices = np.where(y == cls)[0]

            n_needed = max_count - count

            chosen = rng.choice(cls_indices, size=n_needed, replace=True)

            X_resampled.extend(X[chosen])

            y_resampled.extend(y[chosen])

    X_out = np.array(X_resampled)

    y_out = np.array(y_resampled)

    shuffle = rng.permutation(len(y_out))

    return X_out[shuffle], y_out[shuffle]

def random_undersample(X, y, seed=42):

    rng = np.random.RandomState(seed)

    classes, counts = np.unique(y, return_counts=True)

    min_count = counts.min()

    X_resampled = []

    y_resampled = []

    for cls in classes:

        cls_indices = np.where(y == cls)[0]

        chosen = rng.choice(cls_indices, size=min_count, replace=False)

        X_resampled.extend(X[chosen])

        y_resampled.extend(y[chosen])

    X_out = np.array(X_resampled)

    y_out = np.array(y_resampled)

    shuffle = rng.permutation(len(y_out))

    return X_out[shuffle], y_out[shuffle]

In [ ]:
```

### Step 4: Logistic regression with class weights

In [ ]:
```python

def sigmoid(z):

    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def logistic_regression_weighted(X, y, weights, lr=0.01, epochs=200):

    n_samples, n_features = X.shape

    w = np.zeros(n_features)

    b = 0.0

    for _ in range(epochs):

        z = X @ w + b

        pred = sigmoid(z)

        error = pred - y

        weighted_error = error * weights

        gradient_w = (X.T @ weighted_error) / n_samples

        gradient_b = np.mean(weighted_error)

        w -= lr * gradient_w

        b -= lr * gradient_b

    return w, b

def compute_class_weights(y):

    classes, counts = np.unique(y, return_counts=True)

    n_samples = len(y)

    n_classes = len(classes)

    weight_map = {}

    for cls, count in zip(classes, counts):

        weight_map[cls] = n_samples / (n_classes * count)

    return np.array([weight_map[yi] for yi in y])

In [ ]:
```

### Step 5: Threshold tuning

In [ ]:
```python

def find_optimal_threshold(y_true, y_probs, metric="f1"):

    best_threshold = 0.5

    best_score = -1.0

    for threshold in np.arange(0.05, 0.96, 0.01):

        y_pred = (y_probs >= threshold).astype(int)

        tp = np.sum((y_pred == 1) & (y_true == 1))

        fp = np.sum((y_pred == 1) & (y_true == 0))

        fn = np.sum((y_pred == 0) & (y_true == 1))

        if metric == "f1":

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0

            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

            score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        elif metric == "recall":

            score = tp / (tp + fn) if (tp + fn) > 0 else 0.0

        elif metric == "precision":

            score = tp / (tp + fp) if (tp + fp) > 0 else 0.0

        if score > best_score:

            best_score = score

            best_threshold = threshold

    return best_threshold, best_score

In [ ]:
```

### Step 6: Evaluation functions

In [ ]:
```python

def confusion_matrix_values(y_true, y_pred):

    tp = np.sum((y_pred == 1) & (y_true == 1))

    tn = np.sum((y_pred == 0) & (y_true == 0))

    fp = np.sum((y_pred == 1) & (y_true == 0))

    fn = np.sum((y_pred == 0) & (y_true == 1))

    return tp, tn, fp, fn

def compute_metrics(y_true, y_pred):

    tp, tn, fp, fn = confusion_matrix_values(y_true, y_pred)

    accuracy = (tp + tn) / (tp + tn + fp + fn)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    denom = np.sqrt(float((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)))

    mcc = (tp * tn - fp * fn) / denom if denom > 0 else 0.0

    return {

        "accuracy": accuracy,

        "precision": precision,

        "recall": recall,

        "f1": f1,

        "mcc": mcc,

    }

In [ ]:
```

### Step 7: Compare all approaches

In [ ]:
```python

X, y = make_imbalanced_data(950, 50, seed=42)

split = int(0.8 * len(y))

X_train, X_test = X[:split], X[split:]

y_train, y_test = y[:split], y[split:]

# Baseline: no treatment

w_base, b_base = logistic_regression_weighted(

    X_train, y_train, np.ones(len(y_train)), lr=0.1, epochs=300

)

probs_base = sigmoid(X_test @ w_base + b_base)

preds_base = (probs_base >= 0.5).astype(int)

# Oversampled

X_over, y_over = random_oversample(X_train, y_train)

w_over, b_over = logistic_regression_weighted(

    X_over, y_over, np.ones(len(y_over)), lr=0.1, epochs=300

)

preds_over = (sigmoid(X_test @ w_over + b_over) >= 0.5).astype(int)

# SMOTE

minority_mask = y_train == 1

X_minority = X_train[minority_mask]

synthetic = smote(X_minority, k=5, n_synthetic=len(y_train) - 2 * int(minority_mask.sum()))

X_smote = np.vstack([X_train, synthetic])

y_smote = np.concatenate([y_train, np.ones(len(synthetic))])

w_sm, b_sm = logistic_regression_weighted(

    X_smote, y_smote, np.ones(len(y_smote)), lr=0.1, epochs=300

)

preds_smote = (sigmoid(X_test @ w_sm + b_sm) >= 0.5).astype(int)

# Class weights

sample_weights = compute_class_weights(y_train)

w_cw, b_cw = logistic_regression_weighted(

    X_train, y_train, sample_weights, lr=0.1, epochs=300

)

probs_cw = sigmoid(X_test @ w_cw + b_cw)

preds_cw = (probs_cw >= 0.5).astype(int)

# Threshold tuning (tune on held-out validation set, not test set)

probs_val = sigmoid(X_val @ w_cw + b_cw)

best_thresh, best_f1 = find_optimal_threshold(y_val, probs_val, metric="f1")

preds_thresh = (probs_cw >= best_thresh).astype(int)

In [ ]:
```

The code file runs all of this in a single script and prints results.

## Exercises

In [ ]:
1. **Borderline-SMOTE**: modify the SMOTE implementation to only generate synthetic samples for minority points that are near the decision boundary (those whose k-nearest neighbors include majority class samples). Compare results with standard SMOTE on a dataset where classes overlap.

2. **Cost matrix optimization**: implement cost-sensitive learning where the cost matrix is a parameter. Create a function that takes a cost matrix and returns optimal predictions that minimize expected cost. Test with different cost ratios (1:10, 1:100, 1:1000) and plot how the precision-recall tradeoff changes.

3. **Threshold calibration**: implement Platt scaling (fit a logistic regression on the model's raw outputs to produce calibrated probabilities). Compare the precision-recall curve before and after calibration. Show that calibration does not change the ranking (AUC stays the same) but makes the probabilities more meaningful.

4. **Ensemble with balanced bagging**: train multiple models, each on a balanced bootstrap sample (all minority + random subset of majority). Average their predictions. Compare this approach against a single model with SMOTE. Measure both performance and variance across runs.

5. **Imbalance ratio experiment**: take a balanced dataset and progressively increase the imbalance ratio (50/50, 70/30, 90/10, 95/5, 99/1). For each ratio, train with and without SMOTE. Plot F1 vs imbalance ratio for both approaches. At what ratio does SMOTE start making a meaningful difference?